# Healthcare Agent Learning Lab 8/10

## Orchestrators and specialist agents

**Suggested time:** 25 minutes  
**Demonstration type:** Scripted or live comparison of architectures

### Learning objectives

- Compare one-agent and multi-agent architectures for the same task.
- Identify when specialization or parallelism justifies coordination overhead.
- Recognize aggregation, disagreement, and handoff failures.

### How to use this notebook

1. **Observe:** run the supplied demonstration and read the visible trajectory.
2. **Exercise:** discuss workflow, evidence, responsibility, and safety—no programming required.
3. **Try it:** edit only the clearly marked values in the exercise cell.
4. **Technical extension:** optional implementation work for technical participants.
5. **Reflect:** record what should be changed before the pattern is used in a real workflow.

> **Mode and safety notice:** Set `LAB_MODE` explicitly to `mock` or `live`; the notebook will stop rather than silently choose a mode. Mock mode is scripted and demonstrates system structure—it is not evidence of model performance. Live mode may vary between runs and may incur API cost. All patients, actions, protocols, and outputs in this curriculum are fictional or synthetic and must not be used for patient care.


## Demonstration
**Goal:** show multi-agent coordination *and* its cost. An **orchestrator** decomposes a request, routes sub-tasks to specialist sub-agents, then **aggregates and verifies** — and we run the *same* task with one well-instrumented agent so the team can feel when the extra machinery is, and isn't, worth it.

The task: prepare the post-discharge anticoagulant follow-up. (Runs in MOCK mode with no API key.)


> ### In plain terms
> Sometimes you split a job across several **specialist** agents with a **coordinator** on top; sometimes **one** well-run agent is simpler and just as good. This runs the *same* task both ways so you can feel the trade-off: more agents buy specialization and a place to double-check the work, but cost more and add more ways to fail.
>
> **The honest default:** start with one well-instrumented agent; add agents only when the work genuinely needs it.
>
> **What you'll see:** top — three specialists each do one narrow job, then a "verify" step checks their story is consistent. Bottom — one agent does the whole thing. Compare the two plans.


In [ ]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (explicit LAB_MODE=mock): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model (explicit LAB_MODE=live): pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
LAB_MODE = os.environ.get("LAB_MODE", "").strip().lower()
if LAB_MODE not in {"mock", "live"}:
    raise RuntimeError("Set LAB_MODE explicitly to 'mock' or 'live' before running this notebook.")
USE_MOCK = LAB_MODE == "mock"
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")
if not USE_MOCK and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LAB_MODE=live requires OPENAI_API_KEY in the Jupyter process environment.")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    usage = getattr(r, "usage", None)
    prompt_details = getattr(usage, "prompt_tokens_details", None) if usage else None
    completion_details = getattr(usage, "completion_tokens_details", None) if usage else None
    reported_cost = getattr(usage, "cost", None) if usage else None
    if reported_cost is None and getattr(usage, "model_extra", None):
        reported_cost = usage.model_extra.get("cost")
    globals()["_LAB_LAST_RESPONSE_META"] = {
        "response_id": getattr(r, "id", None),
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "cached_input_tokens": int(getattr(prompt_details, "cached_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
        "reasoning_tokens": int(getattr(completion_details, "reasoning_tokens", 0) or 0),
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0),
        "reported_cost_usd": float(reported_cost) if reported_cost is not None else None,
    }
    return r.choices[0].message.content

print("Backend:", "MOCK (explicit scripted mode — no model calls)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages, temperature=0):
    "Route by the specialist role named in the system prompt."
    sys = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    if "triage"     in sys: return "urgency=HIGH; INR 4.2 is well above range with recent DVT."
    if "records"    in sys: return "INR 4.2 (range 2.0-3.0); meds: warfarin 5 mg daily, lisinopril 10 mg."
    if "scheduling" in sys: return "Earliest anticoagulation-clinic slot: in 2 days (Thu 10:00)."
    # single-agent path: produce the whole plan at once
    return ("Plan: flag out-of-range INR (4.2) for clinician dose review; bring the anticoagulation "
            "follow-up forward to ~2 days; no order changed by the agent.")


In [ ]:
# LEARNING-LAB OBSERVABILITY WRAPPER
# Records calls, latency, response IDs, token usage, and cost when available.
# Estimated cost uses the explicit teaching-lab rate table below; provider
# billing remains authoritative.
import time as _lab_time

_lab_raw_chat = chat
LAB_CALL_LOG = []
_LAB_LAST_RESPONSE_META = {}
LAB_PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "openai/gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
}

def chat(messages, temperature=0):
    global _LAB_LAST_RESPONSE_META
    _LAB_LAST_RESPONSE_META = {}
    started = _lab_time.perf_counter()
    answer = _lab_raw_chat(messages, temperature=temperature)
    meta = dict(_LAB_LAST_RESPONSE_META)
    input_tokens = int(meta.get("input_tokens", 0) or 0)
    cached_tokens = int(meta.get("cached_input_tokens", 0) or 0)
    output_tokens = int(meta.get("output_tokens", 0) or 0)
    pricing = LAB_PRICING_USD_PER_MILLION.get(MODEL)
    estimated_cost = None
    if pricing:
        estimated_cost = (
            max(input_tokens - cached_tokens, 0) * pricing["input"]
            + cached_tokens * pricing["cached_input"]
            + output_tokens * pricing["output"]
        ) / 1_000_000
    LAB_CALL_LOG.append({
        "mode": "mock" if USE_MOCK else "live",
        "model": MODEL,
        "temperature": temperature,
        "latency_seconds": round(_lab_time.perf_counter() - started, 4),
        "response_id": meta.get("response_id"),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(meta.get("reasoning_tokens", 0) or 0),
        "total_tokens": int(meta.get("total_tokens", 0) or 0),
        "reported_cost_usd": meta.get("reported_cost_usd"),
        "estimated_cost_usd": estimated_cost,
    })
    return answer


### Three specialist sub-agents
Each is just the model with a focused role prompt and a narrow job.


In [ ]:
def specialist(role, instruction, context):
    sys = f"You are the {role} sub-agent. Do only your part. Be terse."
    return chat([{"role":"system","content":sys},
                 {"role":"user","content":f"{instruction}\nContext: {context}"}])

PATIENT = ("Discharged 5 days ago after DVT; on warfarin 5 mg daily and lisinopril 10 mg daily; "
           "INR resulted today at 4.2 (reference range 2.0-3.0).")
def triage_agent(c):    return specialist("triage",     "Assess urgency.", c)
def records_agent(c):   return specialist("records",    "Pull the INR and medications.", c)
def scheduling_agent(c):return specialist("scheduling", "Find the earliest follow-up slot.", c)


### The orchestrator: decompose → dispatch → aggregate → verify
The orchestrator owns the plan and, crucially, a **verification** step over the combined result.


In [ ]:
def one_line(s):  return " ".join(s.split())    # collapse a multi-line reply to one line

def orchestrator(goal, context):
    print("ORCHESTRATOR goal:", goal, "\n")
    results = {"triage":     triage_agent(context),
               "records":    records_agent(context),
               "scheduling": scheduling_agent(context)}
    for k, v in results.items(): print(f"   [{k}] -> {one_line(v)}")

    # verify: is the asserted urgency actually supported by the record? (case-insensitive)
    urgent      = "high" in results["triage"].lower()
    inr_flagged = "4.2"  in results["records"] or "out of range" in results["records"].lower()
    consistent  = urgent == inr_flagged
    print(f"\n   [verify] urgency asserted = {urgent} · out-of-range INR in record = {inr_flagged} "
          f"-> consistent: {consistent}")

    plan = {
        "Urgency":        one_line(results["triage"]),
        "Record":         one_line(results["records"]),
        "Next visit":     one_line(results["scheduling"]),
        "Action":         "draft dose-review flag for clinician; change no order",
        "Verified":       consistent,
    }
    return plan

plan = orchestrator("Prepare the anticoagulant follow-up.", PATIENT)
print("\nAGGREGATED PLAN")
for k, v in plan.items(): print(f"   {k:11}: {v}")


### The same task with one agent
One model, all the context, a single call. Fewer moving parts, no inter-agent messaging.


In [ ]:
single = chat([{"role":"system","content":"You are a clinical follow-up assistant. Produce the plan; change no order."},
               {"role":"user","content":f"Prepare the anticoagulant follow-up. Context: {PATIENT}"}])
print("SINGLE-AGENT PLAN:\n ", single)


### Takeaway
Both produce a usable plan. The multi-agent version buys **separation of concerns, parallelism, and a natural place to verify** — worth it when sub-tasks need different tools, context, or specialization. It also costs more calls, more latency, and more ways to fail (miscommunication between agents is a leading multi-agent failure mode). The honest default from the white paper: **start with one well-instrumented agent and add agents only when the work demands it.** Reach for orchestration for genuinely separable, specialized workloads — not by reflex.

*Try:* give one sub-agent its own tool the others lack (e.g., scheduling calls a calendar API), and the case for splitting gets stronger.


## Exercise

No programming is required for this section.

**Activity:** Decide whether a proposed workflow needs one agent, several specialist agents, or ordinary deterministic software. Identify the accountable owner of the combined output.

### Discussion prompts

- What capability does each specialist have that the others lack?
- Who verifies the combined result?
- Would a single agent with several tools be simpler and safer?

Record the group’s proposed workflow boundary, the evidence needed, and the person or role accountable for the consequential decision.


In [ ]:
# CLINICAL TRY-IT: change these properties and inspect the architecture suggestion.
TASK_IS_SEPARABLE = True
SPECIALISTS_NEED_DIFFERENT_TOOLS = True
COORDINATION_RISK = "high"  # "low" or "high"
if TASK_IS_SEPARABLE and SPECIALISTS_NEED_DIFFERENT_TOOLS and COORDINATION_RISK == "low":
    suggestion = "consider multiple specialists"
else:
    suggestion = "prefer one well-instrumented agent or deterministic workflow"
print("Architecture suggestion:", suggestion)


## Optional technical extension

Give each specialist a typed output contract, inject a disagreement, and implement an aggregator that refuses to finalize unresolved conflicts.

Technical participants should report both successful behavior and at least one intentionally induced failure. Preserve the trace needed to explain why the failure occurred.


In [ ]:
# OBSERVABILITY SUMMARY
print("Model interaction summary")
print("  mode:", "mock" if USE_MOCK else "live")
print("  model:", MODEL)
print("  calls:", len(LAB_CALL_LOG))
print("  total measured latency (s):", round(sum(row["latency_seconds"] for row in LAB_CALL_LOG), 4))
print("  input tokens:", sum(row["input_tokens"] for row in LAB_CALL_LOG))
print("  cached input tokens:", sum(row["cached_input_tokens"] for row in LAB_CALL_LOG))
print("  output tokens:", sum(row["output_tokens"] for row in LAB_CALL_LOG))
print("  reasoning tokens:", sum(row["reasoning_tokens"] for row in LAB_CALL_LOG))
print("  total tokens:", sum(row["total_tokens"] for row in LAB_CALL_LOG))
reported = [row["reported_cost_usd"] for row in LAB_CALL_LOG if row["reported_cost_usd"] is not None]
estimated = [row["estimated_cost_usd"] for row in LAB_CALL_LOG if row["estimated_cost_usd"] is not None]
print("  provider-reported cost (USD):", round(sum(reported), 6) if reported else "unavailable")
print("  estimated cost (USD):", round(sum(estimated), 6) if estimated else "unavailable")
print("  response IDs captured:", sum(bool(row["response_id"]) for row in LAB_CALL_LOG))


## Interpretation boundary

More agents do not imply more intelligence. Require a concrete need—different permissions, tools, context, or parallel work—and measure the added calls, latency, cost, and coordination failures.

### Exit question

In one sentence: **What should the environment enforce, rather than merely asking the model to do?**

Continue to the next notebook only after the group can distinguish the observed system behavior from any claim about clinical correctness.
